EVALUACION DE NAIVE BAYES - IMPLEMENTACION "MANUAL"

In [1]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import pandas as pd
import algoritmos_clasificacion as t

# Hiperparámetros globales
m_values = [0.5, 1, 2]

def evaluar_naive_bayes(ruta_dataset):
    """
    Evalúa un Naive Bayes sobre el dataset en ruta_dataset.
    Devuelve un DataFrame con el resumen de métricas por cada valor de m.
    """
    resultados_bayes = {}

    # Cargar dataset
    S = pd.read_csv(ruta_dataset)

    # Features y target
    #X = S[["hum_carrasco","hum_salto","tmax_carrasco","tmin_carrasco","tmax_salto","tmin_salto",
           #"amplitud_carrasco","amplitud_salto",
           #"dir_viento_carrasco_prom","int_viento_carrasco_prom","int_racha_viento_carrasco_prom",
           #"dir_viento_salto_prom","int_viento_salto_prom","int_racha_viento_salto_prom",
           #"vis_carrasco_prom","vis_salto_prom"]]
    y = S["dia_lluvioso"]
    X = S.drop(columns=["dia_lluvioso","fecha"])

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for m in m_values:
        print(f"\n=== Evaluando Naive Bayes con m={m} ===")
        
        # Crear modelo
        model = t.NaiveBayesClassifier(m=m)
        
        # Validación cruzada en train
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf)
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Métricas en test
        acc_test = accuracy_score(y_test, y_pred)
        f1_test = f1_score(y_test, y_pred, average='binary', pos_label='si')
        recall_test = recall_score(y_test, y_pred, average='binary', pos_label='si')
        cm = confusion_matrix(y_test, y_pred, labels=['si','no'])
        
        # Guardar resultados
        resultados_bayes[m] = {
            "cv_scores": cv_scores,
            "acc_test": acc_test,
            "f1_test": f1_test,
            "recall_test": recall_test,
            "cm": cm
        }

        # Prints intermedios
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)

    # Resumen final en tabla
    resumen = []
    for m, res in resultados_bayes.items():
        resumen.append({
            "m": m,
            "Accuracy CV (mean)": res["cv_scores"].mean(),
            "Accuracy CV (std)": res["cv_scores"].std(),
            "Accuracy test": res["acc_test"],
            "F1 test": res["f1_test"],
            "Recall test": res["recall_test"]
        })

    df_resumen = pd.DataFrame(resumen)
    print("\n=== RESUMEN NAIVE BAYES ===")
    print(df_resumen)

    return df_resumen, resultados_bayes

CON SCIKIT LEARN

In [2]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
import pandas as pd

# Hiperparámetros: al MultinomialNB no le pasamos m, así que solo dejamos alpha
alpha_values = [0.5, 1, 2, 5]

def evaluar_naive_bayes_sklearn(ruta_dataset):
    resultados_bayes = {}

    # Cargar dataset
    S = pd.read_csv(ruta_dataset)

    # Features y target
    X = S.drop(columns=["dia_lluvioso", "fecha"])
    y = S["dia_lluvioso"]

    # Convertir categorías a dummies (one-hot)
    X_enc = pd.get_dummies(X.astype(str))

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_enc, y, test_size=0.2, random_state=42, stratify=y
    )

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for alpha in alpha_values:
        print(f"\n=== Evaluando MultinomialNB con alpha={alpha} ===")
        
        model = MultinomialNB(alpha=alpha)
        
        # Validación cruzada
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf)
        
        # Entrenar en todo el train y evaluar en test
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Métricas en test
        acc_test = accuracy_score(y_test, y_pred)
        f1_test = f1_score(y_test, y_pred, pos_label='si')
        recall_test = recall_score(y_test, y_pred, pos_label='si')
        cm = confusion_matrix(y_test, y_pred, labels=['si','no'])

        resultados_bayes[alpha] = {
            "cv_scores": cv_scores,
            "acc_test": acc_test,
            "f1_test": f1_test,
            "recall_test": recall_test,
            "cm": cm
        }

        # Prints intermedios
        print("Scores por fold:", [round(s,4) for s in cv_scores])
        print(f"Accuracy promedio CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(f"Accuracy en test: {acc_test:.4f}")
        print(f"F1-score en test: {f1_test:.4f}")
        print(f"Recall en test: {recall_test:.4f}")
        print("Matriz de confusión:")
        print(cm)

    # Resumen final en tabla
    resumen = []
    for alpha, res in resultados_bayes.items():
        resumen.append({
            "alpha": alpha,
            "Accuracy CV (mean)": res["cv_scores"].mean(),
            "Accuracy CV (std)": res["cv_scores"].std(),
            "Accuracy test": res["acc_test"],
            "F1 test": res["f1_test"],
            "Recall test": res["recall_test"]
        })

    df_resumen = pd.DataFrame(resumen)
    print("\n=== RESUMEN NAIVE BAYES ===")
    print(df_resumen)

    return df_resumen, resultados_bayes




EVALUACIÓN DE LOS MODELOS

In [ ]:
#modelo1_vent1 con implementacion manual
df_res1_1_m, res_bayes1_1_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent1.csv")
print(df_res1_1_m)


=== Evaluando Naive Bayes con m=0.5 ===
Scores por fold: [0.7155, 0.718, 0.7334, 0.722, 0.7226]
Accuracy promedio CV: 0.7223 ± 0.0061
Accuracy en test: 0.7145
F1-score en test: 0.3426
Recall en test: 0.3356
Matriz de confusión:
[[ 302  598]
 [ 561 2599]]

=== Evaluando Naive Bayes con m=1 ===
Scores por fold: [0.7155, 0.7174, 0.7321, 0.7223, 0.7238]
Accuracy promedio CV: 0.7222 ± 0.0058
Accuracy en test: 0.7140
F1-score en test: 0.3407
Recall en test: 0.3333
Matriz de confusión:
[[ 300  600]
 [ 561 2599]]

=== Evaluando Naive Bayes con m=2 ===
Scores por fold: [0.7152, 0.7164, 0.7315, 0.7223, 0.7244]
Accuracy promedio CV: 0.7220 ± 0.0059
Accuracy en test: 0.7140
F1-score en test: 0.3392
Recall en test: 0.3311
Matriz de confusión:
[[ 298  602]
 [ 559 2601]]

=== RESUMEN NAIVE BAYES ===
     m  Accuracy CV (mean)  Accuracy CV (std)  Accuracy test   F1 test  \
0  0.5            0.722291           0.006124       0.714532  0.342598   
1  1.0            0.722229           0.005823       0.7

In [ ]:
#modelo1_vent1 con implementacion de scikit
df_res1_1_s, res_bayes1_1_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent1.csv")
print(df_res1_1_s)


=== Evaluando MultinomialNB con alpha=0.5 ===
Scores por fold: [0.7158, 0.7167, 0.7328, 0.722, 0.7241]
Accuracy promedio CV: 0.7223 ± 0.0061
Accuracy en test: 0.7143
F1-score en test: 0.3409
Recall en test: 0.3333
Matriz de confusión:
[[ 300  600]
 [ 560 2600]]

=== Evaluando MultinomialNB con alpha=1 ===
Scores por fold: [0.7158, 0.7155, 0.7306, 0.722, 0.7238]
Accuracy promedio CV: 0.7216 ± 0.0056
Accuracy en test: 0.7138
F1-score en test: 0.3383
Recall en test: 0.3300
Matriz de confusión:
[[ 297  603]
 [ 559 2601]]

=== Evaluando MultinomialNB con alpha=2 ===
Scores por fold: [0.7161, 0.7183, 0.734, 0.7241, 0.7257]
Accuracy promedio CV: 0.7236 ± 0.0063
Accuracy en test: 0.7140
F1-score en test: 0.3362
Recall en test: 0.3267
Matriz de confusión:
[[ 294  606]
 [ 555 2605]]

=== Evaluando MultinomialNB con alpha=5 ===
Scores por fold: [0.718, 0.7232, 0.7389, 0.7294, 0.7263]
Accuracy promedio CV: 0.7272 ± 0.0070
Accuracy en test: 0.7182
F1-score en test: 0.3364
Recall en test: 0.3222
Ma

In [ ]:
#modelo1_vent2 con implementacion manual
df_res1_2_m, res_bayes1_2_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent2.csv")
print(df_res1_2_m)

In [ ]:
#modelo1_vent2 con implementacion de scikit
df_res1_2_s, res_bayes1_2_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent2.csv")
print(df_res1_2_s)

In [ ]:
#modelo1_vent3 con implementacion manual
df_res1_3_m, res_bayes1_3_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent3.csv")
print(df_res1_3_m)

In [ ]:
#modelo1_vent3 con implementacion de scikit
df_res1_3_s, res_bayes1_3_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent3.csv")
print(df_res1_3_s)

In [ ]:
#modelo1_vent5 con implementacion manual
df_res1_5_m, res_bayes1_5_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent5.csv")
print(df_res1_5_m)

In [ ]:
#modelo1_vent5 con implementacion de scikit
df_res1_5_s, res_bayes1_5_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent5.csv")
print(df_res1_5_s)

In [ ]:
#modelo1_vent7 con implementacion manual
df_res1_7_m, res_bayes1_7_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent7.csv")
print(df_res1_7_m)

In [ ]:
#modelo1_vent7 con implementacion de scikit
df_res1_7_s, res_bayes1_7_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent7.csv")
print(df_res1_7_s)

In [ ]:
#modelo1_vent10 con implementacion manual
df_res1_10_m, res_bayes1_10_m = evaluar_naive_bayes("../Datos_preprocesados/modelo1_vent10.csv")
print(df_res1_10_m)

In [ ]:
#modelo1_vent10 con implementacion de scikit
df_res1_10_s, res_bayes1_10_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo1_vent10.csv")
print(df_res1_10_s)

In [ ]:
#modelo2_vent1 con implementacion manual
df_res2_1_m, res_bayes2_1_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent1.csv")
print(df_res2_1_m)

In [ ]:
#modelo2_vent1 con implementacion de scikit
df_res2_1_s, res_bayes2_1_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent1.csv")
print(df_res2_1_s)

In [ ]:
#modelo2_vent2 con implementacion manual
df_res2_2_m, res_bayes2_2_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent2.csv")
print(df_res2_2_m)

In [ ]:
#modelo2_vent2 con implementacion de scikit
df_res2_2_s, res_bayes2_2_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent2.csv")
print(df_res2_2_s)

In [ ]:
#modelo2_vent3 con implementacion manual
df_res2_3_m, res_bayes2_3_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent3.csv")
print(df_res2_3_m)

In [ ]:
#modelo2_vent3 con implementacion de scikit
df_res2_3_s, res_bayes2_3_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent3.csv")
print(df_res2_3_s)

In [ ]:
#modelo2_vent5 con implementacion manual
df_res2_5_m, res_bayes2_5_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent5.csv")
print(df_res2_5_m)

In [ ]:
#modelo2_vent5 con implementacion de scikit
df_res2_5_s, res_bayes2_5_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent5.csv")
print(df_res2_5_s)

In [ ]:
#modelo2_vent7 con implementacion manual
df_res2_7_m, res_bayes2_7_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent7.csv")
print(df_res2_7_m)

In [ ]:
#modelo2_vent7 con implementacion de scikit
df_res2_7_s, res_bayes2_7_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent7.csv")
print(df_res2_7_s)

In [ ]:
#modelo2_vent10 con implementacion manual
df_res2_10_m, res_bayes2_10_m = evaluar_naive_bayes("../Datos_preprocesados/modelo2_vent10.csv")
print(df_res2_10_m)

In [ ]:
#modelo2_vent10 con implementacion de scikit
df_res2_10_s, res_bayes2_10_s = evaluar_naive_bayes_sklearn("../Datos_preprocesados/modelo2_vent10.csv")
print(df_res2_10_s)